In [5]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
import xgboost as xgb

from glob import glob
import psi4
from helper_CC_ML_spacial import *

import pyscf
import pyscf.cc
import pyscf.mcscf
import ffsim
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [6]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']
basis = basis_sets[2]
n = 100
model_name = f'optimised_{basis}_best_model_{n}.pkl'
model = joblib.load(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'out', model_name))


In [7]:
# properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag']

In [8]:
with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'out','train_names.txt'), "r") as f:
    lines = f.readlines()

ml_files = [element[:-1] for element in lines]

with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'out','test_names.txt'), "r") as f:
    lines = f.readlines()

ml_files.extend([element[:-1] for element in lines])

molecule_types = ["ammonia", "methane", "ethylene", "ethane", "water", "formaldehyde", "methanol"]
home = os.path.expanduser("~")
data_dir = os.path.join(home, "DDLUCJ", "machine_learning", "data")
# pick one file per type not in ml_files
selected = []
for mol in molecule_types:
    for fname in os.listdir(data_dir):
        if fname.startswith(mol) and fname.endswith(".xyz") and fname not in ml_files:
            selected.append(fname)
            break  # stop after first match

mol_file = selected[4]

In [9]:
with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'data', mol_file),'r') as f:
    text=f.read()

mol = psi4.geometry(text)
print(text)
psi4.core.clean()
psi4.core.be_quiet()

psi4.set_options({'basis': basis,
                  'scf_type':     'pk',
                  'reference':    'rohf',
                  'mp2_type':     'conv',
                  'e_convergence': 1e-8,
                  'd_convergence': 1e-8})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)


O 10 10 10 
H 10.5541 10.1925 10.7596 
H 10.5285 10.0337 9.20182 
units angstrom
symmetry c1

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 41 basis functions.
(41, 41)
(41, 41)
Building initial guess...

..initialized CCSD in 0.599 seconds.



In [13]:
top5 = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']
X= np.vstack([getattr(A,i).flatten() for i in top5]).T
y = model.predict(X)
t2_ml = y.reshape(*A.t2.shape)

In [20]:
y = model.predict(X)  # shape (nocc, nocc, nvirt, nvirt)
t2_ml = y.reshape(*A.t2.shape)
# 2) enforce symmetries used by this helper (spin-adapted t2)
t2_ml = 0.25* (
    t2_ml
    + t2_ml.swapaxes(0, 1)                          # i <-> j
    + t2_ml.swapaxes(2, 3)                          # a <-> b
    + t2_ml.swapaxes(0, 1).swapaxes(2, 3)
)
t2_ml = t2_ml.astype(float, copy=False)


cc = HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
# Baseline (default guess t1=0, t2=MP2)
Ecorr_default = cc.compute_corr_energy()
Etot_default = cc.rhf_e + Ecorr_default

# Replace with ML t2 (keep t1 as your model dictates; 0 if you don’t predict it)
cc.t2 = t2_ml
cc.t1 = np.zeros_like(cc.t1)  # or your ML t1 if you have it
Ecorr_ml = cc.compute_corr_energy()
Etot_ml  = cc.rhf_e + Ecorr_ml



cc = HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
cc.t2 = t2_ml
cc.t1 = np.zeros_like(cc.t1)  # or ML t1
Ecorr = cc.compute_energy()   # full CCSD from ML start



Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 41 basis functions.
(41, 41)
(41, 41)
Building initial guess...

..initialized CCSD in 0.536 seconds.

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 41 basis functions.
(41, 41)
(41, 41)
Building initial guess...

..initialized CCSD in 0.550 seconds.

CCSD Iteration   0: CCSD correlation = -0.121717019482218   dE =  1.21717E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.232244516560911   dE = -1.10527E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.228264371276941   dE =  3.98015E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.229304388130451   dE = -1.04002E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.229202093065633   dE =  1.02295E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.229249713095442   dE = -4.76200E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.229249265666736   dE =  4.47429E-07   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.229249446751054   dE = -1.81084E-07   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.229249753624461   dE = -3.06873E-07   DIIS = 7


In [21]:
print(Etot_ml  - Etot_default)

E_final, steps = cc.rhf_e + Ecorr, cc.steps
print("Final CCSD:", E_final, "iters:", steps)

0.09999627250446963
Final CCSD: -76.27017230397351 iters: 13
